In [2]:
import duckdb
import pandas as pd

# Load the SQL extension
%load_ext sql

# Connect to an in-memory DuckDB instance
%sql duckdb://

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'duckdb://'

In [2]:
import os
os.environ["EVERGREEN_CACHE_DIR_ROOT"] = "~/.cache/evergreen"

from evergreen import SessionContext, col, proportion


ctx = SessionContext.create_optimized()
df = ctx.read_pickle("/Users/alelee/src/sfc-gh-alelee/evergreen/experiments/logs/claim_evaluator/yelp_restaurant_reviews/mcdonalds_mo/rank/ordinal_claim_1/post_sem_op_evg_ref_claude-opus-4-6_claude-opus-4-5_gemini-3-pro_0.pkl")
# df = ctx.read_pickle("/Users/alelee/src/sfc-gh-alelee/evergreen/experiments/logs/claim_evaluator/yelp_restaurant_reviews/mcdonalds_mo/rank/ordinal_claim_2/post_sem_op_evg_opt_claude-haiku-4-5_0.pkl")
print(len(df.collect().rows))
df = df.aggregate([proportion(col("praises_service")).alias("service_praise_prop")], group_by=[col("business_id")]).with_rank(col("service_praise_prop"))

1653


In [11]:
result = df.collect()
[r for r in result.rows][-8:]

[Row(values=('pokplllvDH7X9-n_3IEj_Q', 0.03225806451612903, 42), prov={1: ProvCollection(pos_polynomials=[Prov((('jy1DrA_YcWnDJijQr-WfOQ',), praises_service, True))], neg_polynomials=[Prov((('00g9_av6tvUkvICCIlhSfA',), praises_service, False)), Prov((('80BRONJe5jxyHIIwmqrVdQ',), praises_service, False)), Prov((('FSAjwEYFYjlVZGfL5WFG6g',), praises_service, False)), Prov((('rqZRjf75YdjeK0PFK3aG7A',), praises_service, False)), Prov((('dQ6mHU1nr-z9Nph2K9uymg',), praises_service, False)), Prov((('TI7pGPBlRsz2I738yNFJRA',), praises_service, False)), Prov((('1pJW3XjjNhU0pI1Ca4iD7w',), praises_service, False)), Prov((('a6b_WrWd7umro-EvtecAjA',), praises_service, False)), Prov((('Q2kadwF_Z4CYm-0eNyKrmw',), praises_service, False)), Prov((('J1XwxwIRcAYNonPQzlJWvw',), praises_service, False)), Prov((('mxhnOBvx-gJ7DAIAJuke6A',), praises_service, False)), Prov((('t1yDA47D7nZGt6-WUGjItw',), praises_service, False)), Prov((('LdryxinT0yu-DR3O4QLcAQ',), praises_service, False)), Prov((('ozza-DHgb45nAPC

In [3]:
%%sql
create or replace table yelp_business as 
select *
from read_json_auto('../../../data/yelp_dataset/yelp_academic_dataset_business.json');

create or replace table yelp_review as
select *
from read_json_auto('../../../data/yelp_dataset/yelp_academic_dataset_review.json');

create or replace table yelp_restaurant_business as
select *
from yelp_business
where contains(categories, 'Restaurants');

create or replace table yelp_restaurant_review as
select review_id,
        user_id,
        yelp_review.business_id,
        yelp_restaurant_business.name,
        yelp_restaurant_business.address,
        yelp_restaurant_business.city,
        yelp_restaurant_business.state,
        yelp_restaurant_business.postal_code,
        yelp_restaurant_business.categories,
        yelp_review.stars,
        date,
        text
from yelp_review
inner join yelp_restaurant_business
on yelp_review.business_id = yelp_restaurant_business.business_id;

Running query in 'duckdb://'

Count


In [31]:
%%sql
select count() as cnt
from yelp_restaurant_review
where business_id = 'ZYjbZBJFGk-Tn6cpD-K5eQ';

Running query in 'duckdb://'

cnt
6


In [11]:
%%sql
select business_id,
    count() as cnt 
from yelp_restaurant_review
group by business_id
having cnt >= 2000
order by cnt;

Running query in 'duckdb://'

business_id,cnt
Zi-F-YvyVOK0k5QD7lrLOg,2008
c-iKAO2GBzSKjm7y1Oljcw,2018
2BMk_drsikKWslJCXmQtjQ,2023
hfbZ97Te3T4jeWN6GgsGrQ,2050
iwmW2mgcn2YdirXUHCsgXQ,2064
gVU0U7gNBMQxKTzqmRtdLg,2067
kZ1q0K13tFYG_ZJrVvsJHA,2076
ReV4Q3rEJ8neicQPc6pC0w,2079
75FY8ZQx5nOWP0VFmNvWfw,2083
pSmOH4a3HNNpYM82J5ycLA,2091


In [11]:
%%sql
select *
from yelp_restaurant_review
where review_id = 'N3MEL9VL3d5BWDkRcH2imA';

Running query in 'duckdb://'

review_id,user_id,business_id,name,address,city,state,postal_code,categories,stars,date,text
N3MEL9VL3d5BWDkRcH2imA,Bm6xvB48HZqKHM3kZjCE6A,mhK2uIu7F06ypYdQ5ndmzg,McDonald's,3296 Rider Trl S,Earth City,MO,63045,"Burgers, Coffee & Tea, Food, Fast Food, Restaurants",2.0,2016-08-13 16:10:26,Lame customer service on a Saturday....slow11:01 haven't changed menu boards and I order breakfast. Sorry we're serving lunch...whatever.Going downhill fast.


In [15]:
%%sql
select state, count() as cnt
from yelp_restaurant_review
where contains(name, 'McDonald''s')
group by state
order by cnt desc;

Running query in 'duckdb://'

state,cnt
FL,3340
PA,2929
IN,1991
MO,1813
TN,1643
AZ,1423
NV,1336
LA,1151
NJ,793
IL,510


In [5]:
%%sql
select count(distinct business_id) as cnt
from yelp_restaurant_review
where contains(name, 'McDonald''s') and state = 'MO';

Running query in 'duckdb://'

cnt
62
